# Resume an Existing Atlas

This tutorial shows how to reopen an existing `.sasql` Atlas, inspect the
persisted analysis state, and continue a workflow in a later Python session
without importing the source data again.

Use this tutorial when you have already completed part of an analysis and want
to determine which results are available and which step should be run next.

By the end of this tutorial, you will be able to:

- safely reconnect to an existing Atlas;
- inspect imported data, metadata, and analysis results;
- review the current read index;
- identify the appropriate point from which to resume the workflow;
- continue analysis without unnecessarily recomputing existing results.

## Before You Begin

Resuming an Atlas restores information that has been written to the `.sasql`
database, including expression data, metadata, embeddings, cluster labels, and
other stored analysis results.

Objects that existed only in the previous Python session are not restored
automatically. These may include:

- Python variables other than data stored in the Atlas;
- active data iterators and minibatch generators;
- fitted scikit-learn or PyTorch models that were not saved separately;
- temporary arrays, data frames, and plotting objects.

Save external models and other Python objects separately when they are required
in a later session.


## 1. Verify the Atlas Path

Check that the expected `.sasql` file exists before constructing the `Atlas`
object:


In [15]:
import os
from pathlib import Path
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")


Checking the path first helps prevent a misspelled path from being mistaken for
an existing analysis.


## 2. Open the Atlas

Create a new `Atlas` object using the same database path:


In [16]:
atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)

scAtlasPy connects to the existing database and makes its stored data and
analysis results available in the current Python session.

`db_memory_limit` configures the DuckDB connection created for this session. It
does not modify the expression data or analysis results already stored in the
database.


## 3. Inspect the Atlas Summary

Begin with a general summary:


In [17]:
atlas.describe()

'file_name       : /home/hanxu/scAtlaspy-code-analysis/tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql\ndb_memory_limit : 8GB\ntables          : 17\ntable names     : X_HyS_data, X_HyS_data_filtered, X_HyS_indptr, X_HyS_indptr_filtered, atlas_read_index_meta, kmeans_centers, manual_cluster_annotation, obs, obs_cluster, obsm_X_pca, obsm_X_umap, rank_genes_groups, uns_pca_stats, uns_umap_eval, uns_umap_params, var, varm_PCs\nn_cells         : 2,700\nn_genes         : 32,738'

List the available tables:


In [18]:
atlas.table_names()

['X_HyS_data',
 'X_HyS_data_filtered',
 'X_HyS_indptr',
 'X_HyS_indptr_filtered',
 'atlas_read_index_meta',
 'kmeans_centers',
 'manual_cluster_annotation',
 'obs',
 'obs_cluster',
 'obsm_X_pca',
 'obsm_X_umap',
 'rank_genes_groups',
 'uns_pca_stats',
 'uns_umap_eval',
 'uns_umap_params',
 'var',
 'varm_PCs']

Preview the cell and gene metadata:


In [19]:
atlas.head("obs", n=5)

,atlas_cell_id,atlas_cell_name,filter_cells,cell_total_counts,n_genes_by_counts,total_counts_mt,pct_counts_mt,total_counts_ribo,pct_counts_ribo,scale_factor,kmeans,cell_type_manual,filter_cell_id
0,0,GAACCTGAACGTGT-1,True,4756.0,1491,152.0,3.195963,1826.0,38.393608,2.102607,7,CD4 T,0
1,1,AATCTCTGCTTTAC-1,True,1710.0,668,31.0,1.812865,516.0,30.175438,5.847953,6,NK,1
2,2,GGACCTCTGTAAGA-1,True,2399.0,900,78.0,3.251355,454.0,18.924551,4.168404,2,CD14+ Monocytes,2
3,3,ATAGTCCTAGTGTC-1,True,2205.0,886,47.0,2.131519,450.0,20.408163,4.535147,2,CD14+ Monocytes,3
4,4,GGGATTACGTCTAG-1,True,3001.0,1019,56.0,1.866045,1292.0,43.052315,3.332223,7,CD4 T,4


Check that important metadata fields, such as sample, donor, batch, condition,
cluster, or cell-type annotations, are present.

You can inspect the schemas of the metadata tables with:


In [20]:
atlas.table_info("obs")

,cid,name,type,notnull,dflt_value,pk
0,0,atlas_cell_id,INTEGER,True,None,True
1,1,atlas_cell_name,VARCHAR,False,None,False
2,2,filter_cells,BOOLEAN,False,CAST('f' AS BOOLEAN),False
3,3,cell_total_counts,FLOAT,False,None,False
4,4,n_genes_by_counts,INTEGER,False,None,False
5,5,total_counts_mt,FLOAT,False,None,False
6,6,pct_counts_mt,FLOAT,False,None,False
7,7,total_counts_ribo,FLOAT,False,None,False
8,8,pct_counts_ribo,FLOAT,False,None,False
9,9,scale_factor,FLOAT,False,None,False


In [21]:
atlas.table_info("var")

,cid,name,type,notnull,dflt_value,pk
0,0,atlas_gene_id,USMALLINT,True,None,True
1,1,atlas_gene_name,VARCHAR,False,None,False
2,2,gene_ids,VARCHAR,False,None,False
3,3,filter_genes,BOOLEAN,False,CAST('f' AS BOOLEAN),False
4,4,mt,BOOLEAN,False,None,False
5,5,ribo,BOOLEAN,False,None,False
6,6,gene_total_counts,FLOAT,False,None,False
7,7,n_cells_by_counts,INTEGER,False,None,False
8,8,highly_variable_genes,BOOLEAN,False,None,False
9,9,highly_variable_rank,FLOAT,False,None,False


## 4. Review the Persisted Workflow State

Use `workflow_state()` to summarize common artifacts that indicate how far the
analysis has progressed:


In [22]:
atlas.workflow_state()


,artifact,present,evidence,meaning
0,imported_data,True,obs and var tables,Data have been imported.
1,cell_filter,True,obs.filter_cells,Cell filtering has been computed.
2,gene_filter,True,var.filter_genes,Gene filtering has been computed.
3,highly_variable_genes,True,var.highly_variable_genes,HVG selection has been computed.
4,read_index,True,atlas_read_index_meta table,A read index has been constructed.
5,pca,True,obsm_X_pca and varm_PCs tables,PCA coordinates and loadings have been stored.
6,kmeans,True,obs.kmeans,KMeans cluster labels have been stored.
7,umap,True,obsm_X_umap table,UMAP coordinates have been stored.
8,rank_genes_groups,True,rank_genes_groups table,Marker-gene ranking results have been stored.
9,manual_annotation,True,manual_cluster_annotation table or obs.cell_ty...,Manual annotation results have been stored.


## 5. Inspect the Current Read Index

Many atlas-scale algorithms and streaming workflows operate through a read
index. The read index determines:

- which cells are included;
- which genes are included;
- whether highly variable genes are used;
- which expression field is read.

Inspect the persisted read-index configuration with:


In [23]:
atlas.read_index_info()


,key,value
0,cell_condition,filter_cells
1,gene_condition,filter_genes
2,use_data,data_log1p
3,use_hvg,True


Confirm that its cell selection, gene selection, and expression field match the
analysis you want to continue.

For example, a preprocessing workflow may use:


In [24]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_log1p",
)


build_read_index:   0%|          | 0/2286884 [00:00<?, ?rows/s]

In this example:

- only cells passing `filter_cells` are included;
- only genes passing `filter_genes` are included;
- the selection is further restricted to highly variable genes;
- downstream streaming methods read `data_log1p`.

```{warning}
Do not rebuild the read index unless its current configuration is missing or
does not match the intended analysis.

Changing the read index does not automatically recompute existing PCA,
clustering, UMAP, or model results. Results calculated with an earlier cell
selection, gene selection, or expression field may no longer be consistent with
the new read index and should be recomputed when necessary.
```


## 6. Choose Where to Resume

Use the persisted state to identify the next analysis step.

| Current Atlas state | Typical next action |
|---|---|
| Expression data have been imported, but preprocessing is incomplete | Continue with quality control, filtering, normalization, and feature selection. |
| Preprocessing is complete, but no suitable read index exists | Construct the read index for the intended analysis. |
| The read index is ready, but PCA is missing | Run PCA. |
| PCA is available, but clustering is missing | Run the selected clustering method. |
| PCA is available, but UMAP is missing | Calculate UMAP coordinates. |
| Analysis results are complete | Inspect visualizations, query stored results, annotate cells, or export data. |

Run only the steps that are missing or that need to be recalculated.

For example, if preprocessing and read-index construction are complete but PCA
has not yet been calculated:


In [25]:
# sap.tl.pca(
#     atlas,
#     n_components=50,
#     fit_batches=1000,
# )


If PCA is already available but KMeans clustering is missing:


In [26]:
# sap.tl.kmeans(
#     atlas,
#     n_clusters=10,
#     fit_batches=1000,
# )


If the required upstream results are already present, continue directly to the
corresponding downstream step rather than rerunning the entire workflow.

```{note}
Whether an existing result should be reused depends on more than its presence
in the database. Recompute downstream results when their upstream cell
selection, gene selection, expression representation, or parameters have
changed.
```


## 7. Close the Connection

Close the database connection when the current session is complete:


In [27]:
atlas.close()


Closing the connection does not delete the `.sasql` file or its stored results.
The same Atlas can be reopened in another Python session using its database
path.

## Next Steps

- See {doc}`visualize-analysis-results` to inspect quality-control,
  dimensionality-reduction, clustering, and marker-analysis results.
- See {doc}`query-atlas-with-sql` to explore metadata and analysis results
  directly with SQL.
- Return to the {doc}`../basic/index` if quality control or preprocessing has
  not yet been completed.
